# Testing IMRPhenomXAS_NRTidalv3

## Installation

From the root of the cloned repository, install the requirements and Sage itself:

```bash
python -m pip install -r requirements.txt
python -m pip install -e .
```

Then run this notebook with that Python environment. The README has a conda option as well.

## Generating a single waveform

In [ ]:
# Parameters of one BNS signal
params = {
    "mass1": 1.46,              # solar masses
    "mass2": 1.27,
    "spin1x": 0.0,              # transverse spins: zero for this aligned-spin model
    "spin1y": 0.0,
    "spin1z": 0.02,             # aligned spins
    "spin2x": 0.0,
    "spin2y": 0.0,
    "spin2z": -0.01,
    "distance": 40.0,           # Mpc
    "tc": 1187008882.43,        # geocentric GPS merger time (s)
    "coa_phase": 0.0,           # rad
    "inclination": 2.5,         # rad
    "lambda1": 250.0,           # tidal deformabilities
    "lambda2": 400.0,
    "polarization": 0.0,        # rad
    "ra": 3.446,                # rad
    "dec": -0.408,              # rad
}

In [ ]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

from sage.data.waveform import generate_fd_waveform

sample_rate = 8192.0
delta_f = 1 / 512

# No detectors are given, so this returns the two polarisations, h+ and hx;
# tc, ra, dec and polarization are unused and only take effect with detectors.
hplus, hcross = generate_fd_waveform(
    params, 
    approximant="IMRPhenomXAS_NRTidalv3", 
    delta_f=delta_f,
    sample_rate=sample_rate
)

hplus.shape, hcross.shape

In [ ]:
# Visualise hplus and hcross
import matplotlib.pyplot as plt
import numpy as np

# LAL's grid: bin k is at k * delta_f, from DC; bins below f_lower and above the taper are zero
f = np.arange(hplus.shape[-1]) * delta_f
live = np.abs(hplus[0].numpy()) > 0

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

for ax, h, label in zip(axes, (hplus[0], hcross[0]), (r"Re $\tilde{h}_+(f)$", r"Re $\tilde{h}_\times(f)$")):
    ax.semilogx(f[live], h.numpy()[live].real, lw=0.5)
    ax.set_ylabel(label)
    ax.grid(True, which="both", alpha=0.3)

axes[-1].set_xlabel("Frequency (Hz)")
fig.tight_layout()
plt.show()

Applying the extrinsic parameters requires one to specify the detectors as well. Next, we test it with detectors, which applies each detector's antenna pattern to h+ and hx.

In [ ]:
# With detectors, tc, ra, dec and polarization take effect: each detector's strain is
# F+ h+ + Fx hx at its arrival time, with the merger at the end of the segment by default.
detectors = ["H1", "L1", "V1"]

strain = generate_fd_waveform(
    params,
    approximant="IMRPhenomXAS_NRTidalv3",
    delta_f=delta_f,
    sample_rate=sample_rate,
    detectors=detectors
)

strain.shape

In [ ]:
# Visualise the strain in each detector
live = np.abs(strain[0, 0].numpy()) > 0

fig, axes = plt.subplots(len(detectors), 1, figsize=(10, 3 * len(detectors)), sharex=True)

for ax, name, h in zip(axes, detectors, strain[0]):
    ax.semilogx(f[live], h.numpy()[live].real, lw=0.5)
    ax.set_ylabel(rf"Re $\tilde{{h}}_{{\rm {name}}}(f)$")
    ax.grid(True, which="both", alpha=0.3)
    
axes[-1].set_xlabel("Frequency (Hz)")
fig.tight_layout()
plt.show()

## Generating a set of waveforms

In [ ]:
# Parameters of three BNS signals
signals = [
    {
        "mass1": 1.40, "mass2": 1.35,                                            # solar masses
        "spin1x": 0.0, "spin1y": 0.0, "spin1z": 0.01,                            # transverse spins zero
        "spin2x": 0.0, "spin2y": 0.0, "spin2z": 0.0,
        "distance": 60.0,                                                        # Mpc
        "tc": 1240000000.0,                                                      # geocentric GPS (s)
        "coa_phase": 0.3, "inclination": 0.4,                                    # rad
        "lambda1": 300.0, "lambda2": 350.0,
        "polarization": 1.1, "ra": 2.0, "dec": 0.3,                              # rad
    },
    {
        "mass1": 1.80, "mass2": 1.50,
        "spin1x": 0.0, "spin1y": 0.0, "spin1z": -0.03,
        "spin2x": 0.0, "spin2y": 0.0, "spin2z": 0.02,
        "distance": 100.0,
        "tc": 1240003600.0,
        "coa_phase": 1.7, "inclination": 1.2,
        "lambda1": 100.0, "lambda2": 300.0,
        "polarization": 0.4, "ra": 1.0, "dec": 0.5,
    },
    {
        "mass1": 1.30, "mass2": 1.20,
        "spin1x": 0.0, "spin1y": 0.0, "spin1z": 0.04,
        "spin2x": 0.0, "spin2y": 0.0, "spin2z": -0.02,
        "distance": 150.0,
        "tc": 1240007200.0,
        "coa_phase": 4.0, "inclination": 2.2,
        "lambda1": 800.0, "lambda2": 1000.0,
        "polarization": 2.5, "ra": 5.0, "dec": -1.0,
    },
]

In [ ]:
from sage.data.waveform import generate_fd_waveform

# A list of parameter dicts gives one waveform per dict, in one batch
strains = generate_fd_waveform(
    signals,
    approximant="IMRPhenomXAS_NRTidalv3",
    delta_f=1 / 512,
    sample_rate=8192.0,
    detectors=["H1", "L1", "V1"]
)

strains.shape

## Sampling waveforms from a prior

In [ ]:
import yaml
from sage.data.waveform.generate import load_waveform_prior

# The bns_standard preset, as Sage reads and checks it
prior = load_waveform_prior("bns_standard")

# Uncomment if you'd like to view the YAML representation of the prior preset
# print(yaml.safe_dump(prior, sort_keys=False))

In [ ]:
import torch
from sage.data.waveform import prior_sampler

# Draw from it with Sage's prior sampler: constraints and transforms applied, one column per parameter
sampler = prior_sampler(prior, seed=0, device="cpu", dtype=torch.float64)
draws = sampler(100_000)
samples = {name: draws[:, k].numpy() for name, k in sampler.param_index.items()}

In [ ]:
# One histogram per parameter
import matplotlib.pyplot as plt

fig, axes = plt.subplots(5, 5, figsize=(16, 14))
for ax, (name, values) in zip(axes.flat, samples.items()):
    ax.hist(values, bins=60, histtype="step")
    ax.set_title(name)
    ax.set_yticks([])
fig.tight_layout()
plt.show()

In [ ]:
from sage.data.waveform import sample_fd_waveform

# Draw four signals from the same preset and make their waveforms in H1, L1 and V1;
# delta_f = 1/512 covers the lightest systems in this prior (1 + 1 Msun needs 512 s from 20 Hz)
waveforms, parameters = sample_fd_waveform(
    num_waveforms=4,
    prior="bns_standard",
    approximant="IMRPhenomXAS_NRTidalv3",
    delta_f=1 / 512,
    detectors=["H1", "L1", "V1"],
    seed=0
)

waveforms.shape

## Sage vs LAL comparison

The match is the normalised inner product maximised over coalescence time and phase (PyCBC's `match`), with a flat PSD from 20 Hz, and the mismatch is 1 - match. Each point takes the worse of h+ and hx. The mismatches are plotted as they are, unclipped.

In [ ]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

import numpy as np
import torch
from sage.data.waveform import prior_sampler
from sage.data.waveform.generate import load_waveform_prior

# Draw the signals to compare from the bns_standard preset
num_waveforms = 1000
sampler = prior_sampler(load_waveform_prior("bns_standard"), seed=1, device="cpu", dtype=torch.float64)
draws = sampler(num_waveforms)
drawn = {name: draws[:, k].numpy() for name, k in sampler.param_index.items()}

In [ ]:
import lal
import lalsimulation as lalsim
import pycbc.filter
import pycbc.types
from sage.data.waveform import generate_fd_waveform
from sage.data.waveform.waveform_utils import WAVEFORM_PARAMETERS

approximant = "IMRPhenomXAS_NRTidalv3"
delta_f, f_lower, f_ref, sample_rate = 1 / 512, 20.0, 20.0, 4096.0


def lal_fd_waveform(row):
    """LAL's h+ and hx for one signal, with multibanding off (PhenomXHMThresholdMband = 0)."""
    extra = lal.CreateDict()
    lalsim.SimInspiralWaveformParamsInsertTidalLambda1(extra, row["lambda1"])
    lalsim.SimInspiralWaveformParamsInsertTidalLambda2(extra, row["lambda2"])
    lalsim.SimInspiralWaveformParamsInsertPhenomXHMThresholdMband(extra, 0)
    hp, hc = lalsim.SimInspiralChooseFDWaveform(
        row["mass1"] * lal.MSUN_SI, row["mass2"] * lal.MSUN_SI,
        row["spin1x"], row["spin1y"], row["spin1z"], row["spin2x"], row["spin2y"], row["spin2z"],
        row["distance"] * 1e6 * lal.PC_SI, row["inclination"], row["coa_phase"], 0.0, 0.0, 0.0,
        delta_f, f_lower, sample_rate / 2, f_ref, extra, lalsim.IMRPhenomXAS_NRTidalv3)
    return hp.data.data, hc.data.data


def mismatch(a, b):
    """1 - match, PyCBC's match with a flat PSD from f_lower."""
    a, b = (pycbc.types.FrequencySeries(h, delta_f=delta_f) for h in (a, b))
    return 1.0 - pycbc.filter.match(a, b, low_frequency_cutoff=f_lower)[0]


# The waveform parameters of each signal; the drawn spin components chi1x... stay out
rows = [{name: float(drawn[name][i]) if name in drawn else 0.0 for name in WAVEFORM_PARAMETERS}
        for i in range(num_waveforms)]

# Sage in batches of 25, LAL one by one; each point keeps the worse of h+ and hx
mismatches = np.empty(num_waveforms)
for start in range(0, num_waveforms, 25):
    batch = rows[start:start + 25]
    hp, hc = generate_fd_waveform(batch, approximant=approximant, delta_f=delta_f, sample_rate=sample_rate,
                                  f_lower=f_lower, f_ref=f_ref, numpy=True)
    for k, row in enumerate(batch):
        lal_hp, lal_hc = lal_fd_waveform(row)
        mismatches[start + k] = max(mismatch(hp[k], lal_hp), mismatch(hc[k], lal_hc))

print(f"mismatch: median {np.median(mismatches):.1e}, max {mismatches.max():.1e}")

In [ ]:
# Corner plot over the intrinsic parameters: each hexagon shows the worst mismatch of its waveforms
from sage.plotting import plot_corner_hexbin

intrinsic = ["mass1", "mass2", "spin1z", "spin2z", "lambda1", "lambda2"]
fig = plot_corner_hexbin(
    points={name: drawn[name] for name in intrinsic},
    values=mismatches,
    reduce=np.max,
    colorbar_label="mismatch"
)